In [1]:
%%writefile main.py
import math
import time
from collections import defaultdict, namedtuple
from dataclasses import dataclass, field

# ============================================================
# Shared Configuration & Unified Multipliers
# ============================================================

BOARD = 100.0
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5  # Tighter margin, prevents false-positives
ROTATION_LIMIT = 50.0
TOTAL_STEPS = 500
SIM_HORIZON = 110
HORIZON = SIM_HORIZON

EARLY_TURN_LIMIT = 40
OPENING_TURN_LIMIT = 80
LATE_REMAINING_TURNS = 70          
VERY_LATE_REMAINING_TURNS = 25
TOTAL_WAR_REMAINING_TURNS = 55     

INNER_HUB_RADIUS_LIMIT = 35.0
HUB_RESERVE_INNER = 0.35
HUB_RESERVE_OUTER = 0.20
ORBITAL_ARC_TOLERANCE = 12.0
ORBITAL_ARC_BONUS = 1.15
LEAPFROG_FOB_BONUS = 1.25

SAFE_NEUTRAL_MARGIN = 2
CONTESTED_NEUTRAL_MARGIN = 2
INTERCEPT_TOLERANCE = 1
SAFE_OPENING_PROD_THRESHOLD = 4
ROTATING_OPENING_MAX_TURNS = 13

ATTACK_COST_TURN_WEIGHT = 0.50
SNIPE_COST_TURN_WEIGHT = 0.42
INDIRECT_VALUE_SCALE = 0.15
INDIRECT_FRIENDLY_WEIGHT = 0.35
INDIRECT_NEUTRAL_WEIGHT = 0.9
INDIRECT_ENEMY_WEIGHT = 1.25

STATIC_NEUTRAL_VALUE_MULT = 1.4
STATIC_HOSTILE_VALUE_MULT = 1.65
ROTATING_OPENING_VALUE_MULT = 0.9
HOSTILE_TARGET_VALUE_MULT = 2.05   
OPENING_HOSTILE_TARGET_VALUE_MULT = 1.55
SAFE_NEUTRAL_VALUE_MULT = 1.2
CONTESTED_NEUTRAL_VALUE_MULT = 0.7
EARLY_NEUTRAL_VALUE_MULT = 1.2
COMET_VALUE_MULT = 0.65
SNIPE_VALUE_MULT = 1.12
SWARM_VALUE_MULT = 1.05
CRASH_EXPLOIT_VALUE_MULT = 1.18
FINISHING_HOSTILE_VALUE_MULT = 1.3  
BEHIND_ROTATING_NEUTRAL_VALUE_MULT = 0.92

EXPOSED_PLANET_VALUE_MULT = 2.0    
WEAKEST_ENEMY_VALUE_MULT_4P = 1.5   
WEAKEST_ENEMY_VALUE_MULT_2P = 1.25  
GANG_UP_VALUE_MULT = 1.4            
GANG_UP_POST_BATTLE_DELAY = 2       
GANG_UP_ETA_WINDOW = 4              

NEUTRAL_MARGIN_BASE = 2
NEUTRAL_MARGIN_PROD_WEIGHT = 2
NEUTRAL_MARGIN_CAP = 8
HOSTILE_MARGIN_BASE = 3
HOSTILE_MARGIN_PROD_WEIGHT = 2
HOSTILE_MARGIN_CAP = 12
STATIC_TARGET_MARGIN = 4
CONTESTED_TARGET_MARGIN = 5
FOUR_PLAYER_TARGET_MARGIN = 2       
LONG_TRAVEL_MARGIN_START = 18
LONG_TRAVEL_MARGIN_DIVISOR = 3
LONG_TRAVEL_MARGIN_CAP = 8
COMET_MARGIN_RELIEF = 6
FINISHING_HOSTILE_SEND_BONUS = 3

STATIC_TARGET_SCORE_MULT = 1.18
EARLY_STATIC_NEUTRAL_SCORE_MULT = 1.25
FOUR_PLAYER_ROTATING_NEUTRAL_SCORE_MULT = 0.84
DENSE_STATIC_NEUTRAL_COUNT = 4
DENSE_ROTATING_NEUTRAL_SCORE_MULT = 0.86
SNIPE_SCORE_MULT = 1.12
SWARM_SCORE_MULT = 1.06
CRASH_EXPLOIT_SCORE_MULT = 1.05

LATE_CAPTURE_BUFFER = 5
VERY_LATE_CAPTURE_BUFFER = 3

DEFENSE_SEND_MARGIN_BASE = 1
DEFENSE_SEND_MARGIN_PROD_WEIGHT = 1

REAR_SOURCE_MIN_SHIPS = 16
REAR_DISTANCE_RATIO = 1.25
REAR_STAGE_PROGRESS = 0.78
REAR_SEND_RATIO_TWO_PLAYER = 0.62
REAR_SEND_RATIO_FOUR_PLAYER = 0.60  
REAR_SEND_MIN_SHIPS = 10
REAR_MAX_TRAVEL_TURNS = 40

PARTIAL_SOURCE_MIN_SHIPS = 6
MULTI_SOURCE_TOP_K = 5
MULTI_SOURCE_ETA_TOLERANCE = 2
MULTI_SOURCE_PLAN_PENALTY = 0.97
HOSTILE_SWARM_ETA_TOLERANCE = 1
THREE_SOURCE_SWARM_ENABLED = True
THREE_SOURCE_MIN_TARGET_SHIPS = 20
THREE_SOURCE_ETA_TOLERANCE = 1
THREE_SOURCE_PLAN_PENALTY = 0.93

PROACTIVE_DEFENSE_HORIZON = 12
PROACTIVE_DEFENSE_RATIO = 0.28      
MULTI_ENEMY_PROACTIVE_HORIZON = 14
MULTI_ENEMY_PROACTIVE_RATIO = 0.35  
MULTI_ENEMY_STACK_WINDOW = 4        
REACTION_SOURCE_TOP_K_MY = 4
REACTION_SOURCE_TOP_K_ENEMY = 4
PROACTIVE_ENEMY_TOP_K = 3

CRASH_EXPLOIT_ENABLED = True
CRASH_EXPLOIT_MIN_TOTAL_SHIPS = 7   
CRASH_EXPLOIT_ETA_WINDOW = 3        
CRASH_EXPLOIT_POST_CRASH_DELAY = 1

LATE_IMMEDIATE_SHIP_VALUE = 0.75
WEAK_ENEMY_THRESHOLD = 110          
ELIMINATION_BONUS = 55.0            

BEHIND_DOMINATION = -0.20
AHEAD_DOMINATION = 0.15
FINISHING_DOMINATION = 0.28
FINISHING_PROD_RATIO = 1.15
AHEAD_ATTACK_MARGIN_BONUS = 0.12
BEHIND_ATTACK_MARGIN_PENALTY = 0.05
FINISHING_ATTACK_MARGIN_BONUS = 0.12

DOOMED_EVAC_TURN_LIMIT = 24
DOOMED_MIN_SHIPS = 8

SOFT_ACT_DEADLINE = 0.82
HEAVY_PHASE_MIN_TIME = 0.16
HEAVY_ROUTE_PLANET_LIMIT = 32

# ============================================================
# Shared Types
# ============================================================

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet = namedtuple("Fleet",["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

@dataclass(frozen=True)
class ShotOption:
    score: float
    src_id: int
    target_id: int
    angle: float
    turns: int
    needed: int
    send_cap: int
    mission: str = "capture"
    anchor_turn: int | None = None

@dataclass
class Mission:
    kind: str
    score: float
    target_id: int
    turns: int
    options: list[ShotOption] = field(default_factory=list)

# ============================================================
# Frame-Perfect Physics & Collision Engine
# ============================================================

def dist(ax, ay, bx, by): 
    return math.hypot(ax - bx, ay - by)

def orbital_radius(planet): 
    return dist(planet.x, planet.y, CENTER_X, CENTER_Y)

def is_static_planet(planet): 
    return orbital_radius(planet) + planet.radius >= ROTATION_LIMIT

def fleet_speed(ships):
    if ships <= 1: return 1.0
    ratio = max(0.0, min(1.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

def point_to_segment_distance(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    seg_len_sq = dx * dx + dy * dy
    if seg_len_sq <= 1e-9: return dist(px, py, x1, y1)
    t = max(0.0, min(1.0, ((px - x1) * dx + (py - y1) * dy) / seg_len_sq))
    return dist(px, py, x1 + t * dx, y1 + t * dy)

def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    return point_to_segment_distance(CENTER_X, CENTER_Y, x1, y1, x2, y2) <= SUN_R + safety

def comet_remaining_life(planet_id, comets):
    for group in comets:
        pids = group.get("planet_ids",[])
        if planet_id not in pids: continue
        idx = pids.index(planet_id)
        paths = group.get("paths",[])
        if idx < len(paths):
            return max(0, len(paths[idx]) - group.get("path_index", 0))
    return 0

def get_target_pos_at_turn(target, turn_idx, ang_vel, comets, comet_ids):
    """Calculates EXACT geometric position of a moving target at a specific discrete turn."""
    if target.id in comet_ids:
        for group in comets:
            pids = group.get("planet_ids",[])
            if target.id in pids:
                idx = pids.index(target.id)
                paths = group.get("paths",[])
                f_idx = group.get("path_index", 0) + turn_idx
                if idx < len(paths) and 0 <= f_idx < len(paths[idx]):
                    return paths[idx][f_idx][0], paths[idx][f_idx][1]
        return None
    
    orb_r = dist(target.x, target.y, CENTER_X, CENTER_Y)
    if orb_r + target.radius >= ROTATION_LIMIT:
        return target.x, target.y # Static planet
    
    curr_ang = math.atan2(target.y - CENTER_Y, target.x - CENTER_X)
    new_ang = curr_ang + ang_vel * turn_idx
    return CENTER_X + orb_r * math.cos(new_ang), CENTER_Y + orb_r * math.sin(new_ang)

def calculate_intercept(src, target, ships, ang_vel, comets, comet_ids):
    """
    Frame-perfect solver. It simulates moving forward through time and finds the exact frame 
    the fleet's segment physically intersects the planet.
    """
    speed = fleet_speed(max(1, ships))
    sx, sy, sr = src.x, src.y, src.radius
    tr = target.radius
    
    for N in range(1, 150):
        pos = get_target_pos_at_turn(target, N - 1, ang_vel, comets, comet_ids)
        if pos is None:
            break # Volatile target (comet) disappeared
        tx, ty = pos
        
        D = dist(tx, ty, sx, sy)
        # Fleet needs to reach the edge of the target planet
        travel_dist = max(0.0, D - sr - tr)
        
        N_calc = math.ceil(travel_dist / speed)
        if N_calc <= N: # Impact is physically possible on or before this turn!
            angle = math.atan2(ty - sy, tx - sx)
            # Verify sun avoidance along the exact final path segment
            L_x = sx + math.cos(angle) * sr
            L_y = sy + math.sin(angle) * sr
            E_x = tx - math.cos(angle) * tr
            E_y = ty - math.sin(angle) * tr
            if not segment_hits_sun(L_x, L_y, E_x, E_y, safety=SUN_SAFETY):
                return angle, N
    return None

def get_fleet_arrival(fleet, ang_vel, comets, comet_ids, planet_objs, horizon):
    """
    Dynamically predicts collisions for already-flying fleets against moving planets. 
    (Fixes the 'Stationary Ledger' bug).
    """
    fx, fy = fleet.x, fleet.y
    dx, dy = math.cos(fleet.angle), math.sin(fleet.angle)
    speed = fleet_speed(fleet.ships)
    
    for N in range(1, horizon + 1):
        seg_start_x = fx + dx * speed * (N - 1)
        seg_start_y = fy + dy * speed * (N - 1)
        seg_end_x = fx + dx * speed * N
        seg_end_y = fy + dy * speed * N
        
        # Sun collision
        if segment_hits_sun(seg_start_x, seg_start_y, seg_end_x, seg_end_y, safety=0.0):
            return None, None
            
        best_dist = float('inf')
        hit_planet = None
        
        # Game Engine Step 5: Movement collision (hitting the planet's previous frame position)
        for p in planet_objs:
            pos = get_target_pos_at_turn(p, N - 1, ang_vel, comets, comet_ids)
            if not pos: continue
            px, py = pos
            if point_to_segment_distance(px, py, seg_start_x, seg_start_y, seg_end_x, seg_end_y) <= p.radius:
                d = dist(px, py, seg_start_x, seg_start_y)
                if d < best_dist:
                    best_dist = d
                    hit_planet = p
                    
        if hit_planet:
            return hit_planet, N
            
        # Out of bounds
        if seg_end_x < 0 or seg_end_x > BOARD or seg_end_y < 0 or seg_end_y > BOARD:
            return None, None
            
        # Game Engine Step 6: Rotation Sweep collision (the planet sweeps into the fleet)
        for p in planet_objs:
            if not is_static_planet(p) or p.id in comet_ids:
                pos = get_target_pos_at_turn(p, N, ang_vel, comets, comet_ids)
                if not pos: continue
                px, py = pos
                if dist(px, py, seg_end_x, seg_end_y) <= p.radius:
                    return p, N
                    
    return None, None

def build_arrival_ledger(fleets, planets, ang_vel, comets, comet_ids, horizon):
    ledger = {p.id:[] for p in planets}
    for f in fleets:
        tgt, eta = get_fleet_arrival(f, ang_vel, comets, comet_ids, planets, horizon)
        if tgt: ledger[tgt.id].append((eta, f.owner, int(f.ships)))
    return ledger

# ============================================================
# World Model & Mechanics
# ============================================================

def resolve_arrival_event(owner, garrison, arrivals):
    by_owner = defaultdict(int)
    for _, a_owner, ships in arrivals: by_owner[a_owner] += ships
    if not by_owner: return owner, max(0.0, garrison)

    sorted_players = sorted(by_owner.items(), key=lambda i: i[1], reverse=True)
    top_owner, top_ships = sorted_players[0]
    if len(sorted_players) > 1:
        sec_ships = sorted_players[1][1]
        surv_owner, surv_ships = (-1, 0) if top_ships == sec_ships else (top_owner, top_ships - sec_ships)
    else:
        surv_owner, surv_ships = top_owner, top_ships

    if surv_ships <= 0: return owner, max(0.0, garrison)
    if owner == surv_owner: return owner, garrison + surv_ships
    garrison -= surv_ships
    return (surv_owner, -garrison) if garrison < 0 else (owner, garrison)

def normalize_arrivals(arrivals, horizon):
    return sorted([(max(1, int(math.ceil(t))), o, int(s)) for t, o, s in arrivals if s > 0 and max(1, int(math.ceil(t))) <= horizon])

def simulate_planet_timeline(planet, arrivals, player, horizon):
    horizon = max(0, int(math.ceil(horizon)))
    by_turn = defaultdict(list)
    for item in normalize_arrivals(arrivals, horizon): by_turn[item[0]].append(item)

    owner, garrison = planet.owner, float(planet.ships)
    owner_at, ships_at = {0: owner}, {0: max(0.0, garrison)}
    min_owned = garrison if owner == player else 0.0
    first_enemy, fall_turn = None, None

    for turn in range(1, horizon + 1):
        if owner != -1: garrison += planet.production
        group = by_turn.get(turn,[])
        prev_owner = owner
        if group:
            if prev_owner == player and first_enemy is None and any(i[1] not in (-1, player) for i in group):
                first_enemy = turn
            owner, garrison = resolve_arrival_event(owner, garrison, group)
            if prev_owner == player and owner != player and fall_turn is None:
                fall_turn = turn

        owner_at[turn], ships_at[turn] = owner, max(0.0, garrison)
        if owner == player: min_owned = min(min_owned, garrison)

    holds_full, keep_needed = True, 0
    if planet.owner == player:
        def survives_with_keep(keep):
            s_owner, s_garrison = planet.owner, float(keep)
            for turn in range(1, horizon + 1):
                if s_owner != -1: s_garrison += planet.production
                if by_turn.get(turn):
                    s_owner, s_garrison = resolve_arrival_event(s_owner, s_garrison, by_turn[turn])
                    if s_owner != player: return False
            return s_owner == player

        if survives_with_keep(int(planet.ships)):
            lo, hi = 0, int(planet.ships)
            while lo < hi:
                mid = (lo + hi) // 2
                if survives_with_keep(mid): hi = mid
                else: lo = mid + 1
            keep_needed = lo
        else:
            holds_full, keep_needed = False, int(planet.ships)

    return {"owner_at": owner_at, "ships_at": ships_at, "keep_needed": keep_needed,
            "min_owned": max(0, int(math.floor(min_owned))) if planet.owner == player else 0,
            "first_enemy": first_enemy, "fall_turn": fall_turn, "holds_full": holds_full, "horizon": horizon}

def state_at_timeline(timeline, arrival_turn):
    t = min(max(0, int(math.ceil(arrival_turn))), timeline["horizon"])
    return timeline["owner_at"].get(t, timeline["owner_at"][timeline["horizon"]]), max(0.0, timeline["ships_at"].get(t, timeline["ships_at"][timeline["horizon"]]))

def count_players(planets, fleets):
    return max(2, len(set(p.owner for p in planets if p.owner != -1) | set(f.owner for f in fleets)))

def nearest_distance_to_set(px, py, planets):
    return min((dist(px, py, p.x, p.y) for p in planets), default=10**9)

def indirect_features(planet, planets, player):
    f, n, e = 0.0, 0.0, 0.0
    for o in planets:
        if o.id == planet.id: continue
        d = dist(planet.x, planet.y, o.x, o.y)
        if d >= 1:
            val = o.production / (d + 12.0)
            if o.owner == player: f += val
            elif o.owner == -1: n += val
            else: e += val
    return f, n, e

def detect_exposed_enemy_planets(fleets, enemy_planets):
    exposed = set()
    for planet in enemy_planets:
        outbound = sum(int(f.ships) for f in fleets if f.owner == planet.owner and getattr(f, 'from_planet_id', -1) == planet.id and f.ships >= 5)
        if outbound >= 12 and outbound >= planet.ships * 0.8:
            exposed.add(planet.id)
    return exposed

def _compute_weakest_enemy(enemy_planets, owner_strength, owner_production):
    enemy_owners = set(p.owner for p in enemy_planets)
    if not enemy_owners: return None
    return min(enemy_owners, key=lambda o: owner_strength.get(o, 0) + owner_production.get(o, 0) * 15)

def detect_enemy_planet_battles(arrivals_by_planet, enemy_planets, player):
    battles = []
    for target in enemy_planets:
        attacks =[(int(math.ceil(eta)), o, int(s)) for eta, o, s in arrivals_by_planet.get(target.id, []) if o not in (-1, player) and o != target.owner and s > 0]
        for eta, attacker_owner, attacker_ships in attacks:
            garrison_at_eta = target.ships + target.production * eta
            post_ships = max(0, attacker_ships - garrison_at_eta) if attacker_ships > garrison_at_eta else max(0, garrison_at_eta - attacker_ships)
            if post_ships < 25: 
                battles.append({"target_id": target.id, "battle_turn": eta, "post_owner": attacker_owner if attacker_ships > garrison_at_eta else target.owner, "post_ships": post_ships})
    return battles


class WorldModel:
    def __init__(self, player, step, planets, fleets, ang_vel, comets, comet_ids):
        self.player, self.step = player, step
        self.planets, self.fleets = planets, fleets
        self.ang_vel = ang_vel
        self.comets, self.comet_ids = comets, set(comet_ids)

        self.planet_by_id = {p.id: p for p in planets}
        self.my_planets = [p for p in planets if p.owner == player]
        self.enemy_planets =[p for p in planets if p.owner not in (-1, player)]
        self.neutral_planets = [p for p in planets if p.owner == -1]
        self.static_neutral_planets =[p for p in self.neutral_planets if is_static_planet(p)]

        my_radii =[orbital_radius(p) for p in self.my_planets]
        self.avg_radius = sum(my_radii) / len(my_radii) if my_radii else 50.0
        self.is_inner_hub = self.avg_radius < INNER_HUB_RADIUS_LIMIT
        self.is_outer_hub = self.avg_radius >= INNER_HUB_RADIUS_LIMIT

        self.num_players = count_players(planets, fleets)
        self.remaining_steps = max(1, TOTAL_STEPS - step)
        self.is_early = step < EARLY_TURN_LIMIT
        self.is_opening = step < OPENING_TURN_LIMIT
        self.is_late = self.remaining_steps < LATE_REMAINING_TURNS
        self.is_very_late = self.remaining_steps < VERY_LATE_REMAINING_TURNS
        self.is_total_war = self.remaining_steps < TOTAL_WAR_REMAINING_TURNS
        self.is_four_player = self.num_players >= 4

        self.owner_strength, self.owner_production = defaultdict(int), defaultdict(int)
        for p in planets:
            if p.owner != -1:
                self.owner_strength[p.owner] += int(p.ships)
                self.owner_production[p.owner] += int(p.production)
        for f in fleets: self.owner_strength[f.owner] += int(f.ships)

        self.my_total = self.owner_strength.get(player, 0)
        self.enemy_total = sum(s for o, s in self.owner_strength.items() if o != player)
        self.max_enemy_strength = max((s for o, s in self.owner_strength.items() if o != player), default=0)
        self.my_prod = self.owner_production.get(player, 0)
        self.enemy_prod = sum(p for o, p in self.owner_production.items() if o != player)

        self._weakest_enemy = _compute_weakest_enemy(self.enemy_planets, self.owner_strength, self.owner_production)
        self._weakest_enemy_strength = self.owner_strength.get(self._weakest_enemy, 0) if self._weakest_enemy else 0
        self.exposed_planet_ids = detect_exposed_enemy_planets(fleets, self.enemy_planets)
        
        # FIXED: Arrivals Ledger now understands moving targets
        self.arrivals_by_planet = build_arrival_ledger(fleets, planets, ang_vel, comets, comet_ids, HORIZON)
        self.base_timeline = {p.id: simulate_planet_timeline(p, self.arrivals_by_planet[p.id], player, HORIZON) for p in planets}
        
        self.keep_needed_map = {p.id: self.base_timeline[p.id]["keep_needed"] for p in planets}
        self.min_owned_map = {p.id: self.base_timeline[p.id]["min_owned"] for p in planets}
        self.first_enemy_map = {p.id: self.base_timeline[p.id]["first_enemy"] for p in planets}
        self.fall_turn_map = {p.id: self.base_timeline[p.id]["fall_turn"] for p in planets}
        self.holds_full_map = {p.id: self.base_timeline[p.id]["holds_full"] for p in planets}
        self.indirect_feature_map = {p.id: indirect_features(p, planets, player) for p in planets}

        self.shot_cache, self.probe_candidate_cache, self.best_probe_cache = {}, {}, {}
        self.exact_need_cache = {}
        
        self.total_visible_ships = sum(int(p.ships) for p in planets) + sum(int(f.ships) for f in fleets)
        self.total_production = sum(int(p.production) for p in planets)

    def is_static(self, planet_id): return is_static_planet(self.planet_by_id[planet_id])
    def comet_life(self, planet_id): return comet_remaining_life(planet_id, self.comets)
    def source_inventory_left(self, source_id, spent_total): return max(0, int(self.planet_by_id[source_id].ships) - spent_total[source_id])

    def plan_shot(self, src_id, target_id, ships):
        key = (src_id, target_id, int(ships))
        if key not in self.shot_cache:
            self.shot_cache[key] = calculate_intercept(self.planet_by_id[src_id], self.planet_by_id[target_id], key[2], self.ang_vel, self.comets, self.comet_ids)
        return self.shot_cache[key]

    def probe_ship_candidates(self, src_id, target_id, source_cap, hints=()):
        source_cap = max(1, int(source_cap))
        nhints = tuple(int(math.ceil(h)) for h in hints if h is not None)
        key = (src_id, target_id, source_cap, nhints)
        if key in self.probe_candidate_cache: return self.probe_candidate_cache[key]
        
        t_ships = max(1, int(math.ceil(self.planet_by_id[target_id].ships)))
        vals = set(range(1, min(6, source_cap) + 1)) | {source_cap, max(1, source_cap//2), max(1, source_cap//3), min(source_cap, PARTIAL_SOURCE_MIN_SHIPS), min(source_cap, t_ships + 1), min(source_cap, t_ships + 2), min(source_cap, t_ships + 4), min(source_cap, t_ships + 8)}
        for h in nhints:
            for d in (-2, -1, 0, 1, 2):
                if 1 <= max(1, min(source_cap, h)) + d <= source_cap: vals.add(max(1, min(source_cap, h)) + d)
        res = sorted(vals)
        self.probe_candidate_cache[key] = res
        return res

    def best_probe_aim(self, src_id, target_id, source_cap, hints=(), min_turn=None, max_turn=None, anchor_turn=None, max_anchor_diff=None):
        key = (src_id, target_id, max(1, int(source_cap)), tuple(hints), min_turn, max_turn, anchor_turn, max_anchor_diff)
        if key in self.best_probe_cache: return self.best_probe_cache[key]
        best, best_key = None, None
        for ships in self.probe_ship_candidates(src_id, target_id, source_cap, hints=hints):
            aim = self.plan_shot(src_id, target_id, ships)
            if not aim: continue
            turns = aim[1]
            if (min_turn and turns < min_turn) or (max_turn and turns > max_turn) or (anchor_turn and max_anchor_diff and abs(turns - anchor_turn) > max_anchor_diff): continue
            k = (turns, ships) if anchor_turn is None else (abs(turns - anchor_turn), turns, ships)
            if best_key is None or k < best_key:
                best_key, best = k, (ships, aim)
        self.best_probe_cache[key] = best
        return best

    def projected_state(self, target_id, arrival_turn, planned_commitments=None, extra_arrivals=()):
        pc = planned_commitments or {}
        cutoff = max(1, int(math.ceil(arrival_turn)))
        if not pc.get(target_id) and not extra_arrivals: return state_at_timeline(self.base_timeline[target_id], cutoff)
        arr =[i for i in self.arrivals_by_planet.get(target_id, []) + pc.get(target_id,[]) + list(extra_arrivals) if i[0] <= cutoff]
        return state_at_timeline(simulate_planet_timeline(self.planet_by_id[target_id], arr, self.player, cutoff), cutoff)

    def projected_timeline(self, target_id, horizon, planned_commitments=None, extra_arrivals=()):
        arr = [i for i in self.arrivals_by_planet.get(target_id, []) + (planned_commitments or {}).get(target_id,[]) + list(extra_arrivals) if i[0] <= max(1, int(math.ceil(horizon)))]
        return simulate_planet_timeline(self.planet_by_id[target_id], arr, self.player, max(1, int(math.ceil(horizon))))

    def hold_status(self, target_id, planned_commitments=None, horizon=HORIZON):
        tl = self.projected_timeline(target_id, horizon, planned_commitments) if (planned_commitments or {}).get(target_id) else self.base_timeline[target_id]
        return {k: tl[k] for k in["keep_needed", "min_owned", "first_enemy", "fall_turn", "holds_full"]}

    def _ownership_search_cap(self, eval_turn): 
        return max(32, int(self.total_visible_ships + self.total_production * max(2, eval_turn + 2) + 32))

    def min_ships_to_own_by(self, target_id, eval_turn, attacker_owner, arrival_turn=None, planned_commitments=None, extra_arrivals=(), upper_bound=None):
        pc = planned_commitments or {}
        et, at = max(1, int(math.ceil(eval_turn))), max(1, int(math.ceil(arrival_turn or eval_turn)))
        if at > et: return (max(1, int(upper_bound)) if upper_bound else self._ownership_search_cap(et)) + 1
        nex = tuple((max(1, int(math.ceil(t))), o, int(s)) for t, o, s in extra_arrivals if s > 0 and max(1, int(math.ceil(t))) <= et)
        key = (target_id, et, attacker_owner) if at == et and not pc.get(target_id) and not nex else None
        if key and key in self.exact_need_cache: return self.exact_need_cache[key]
        
        o_bef, s_bef = self.projected_state(target_id, et, pc, nex)
        if o_bef == attacker_owner:
            if key: self.exact_need_cache[key] = 0
            return 0

        def owns_at(s): return self.projected_state(target_id, et, pc, nex + ((at, attacker_owner, int(s)),))[0] == attacker_owner
        
        hi = max(1, int(upper_bound)) if upper_bound else max(1, int(math.ceil(s_bef)) + 1)
        cap = self._ownership_search_cap(et)
        while not upper_bound and hi <= cap and not owns_at(hi): hi *= 2
        if hi > (cap if not upper_bound else hi) and not owns_at(hi): return hi + 1

        lo = 1
        while lo < hi:
            mid = (lo + hi) // 2
            if owns_at(mid): hi = mid
            else: lo = mid + 1
        if key: self.exact_need_cache[key] = lo
        return lo

    def min_ships_to_own_at(self, target_id, arrival_turn, attacker_owner, planned_commitments=None, extra_arrivals=(), upper_bound=None):
        return self.min_ships_to_own_by(target_id, arrival_turn, attacker_owner, arrival_turn, planned_commitments, extra_arrivals, upper_bound)


# ============================================================
# Strategy Engine (Economics & Logistics)
# ============================================================

def stacked_enemy_proactive_keep(planet, world):
    threats = sorted((aim[1], int(e.ships)) for e in world.enemy_planets if (seeded := world.best_probe_aim(e.id, planet.id, max(1, int(e.ships)))) and (aim := seeded[1]) and aim[1] <= MULTI_ENEMY_PROACTIVE_HORIZON)
    if not threats: return 0
    best_stacked, left, running = 0, 0, 0
    for right in range(len(threats)):
        running += threats[right][1]
        while threats[right][0] - threats[left][0] > MULTI_ENEMY_STACK_WINDOW:
            running -= threats[left][1]; left += 1
        best_stacked = max(best_stacked, running)
    return int(best_stacked * MULTI_ENEMY_PROACTIVE_RATIO)

def detect_enemy_crashes(world):
    crashes =[]
    for tgt_id, arr in world.arrivals_by_planet.items():
        ev = sorted([(int(math.ceil(t)), o, int(s)) for t, o, s in arr if o not in (-1, world.player) and s > 0])
        for i in range(len(ev)):
            for j in range(i + 1, len(ev)):
                if ev[i][1] == ev[j][1]: continue
                if abs(ev[i][0] - ev[j][0]) > CRASH_EXPLOIT_ETA_WINDOW: break
                if ev[i][2] + ev[j][2] >= CRASH_EXPLOIT_MIN_TOTAL_SHIPS:
                    crashes.append({"target_id": tgt_id, "crash_turn": max(ev[i][0], ev[j][0])})
    return crashes

def build_policy_state(world, deadline=None):
    exp = lambda: deadline and time.perf_counter() > deadline
    iwm = {t: f * INDIRECT_FRIENDLY_WEIGHT + n * INDIRECT_NEUTRAL_WEIGHT + e * INDIRECT_ENEMY_WEIGHT for t, (f, n, e) in world.indirect_feature_map.items()}
    res, atk, rtm = {}, {}, {}
    
    hub_id = None
    if world.my_planets:
        hub_id = min(world.my_planets, key=lambda p: sum(dist(p.x, p.y, op.x, op.y) for op in world.my_planets)).id

    for t in world.planets:
        if exp(): break
        if t.owner != world.player:
            rtm[t.id] = (
                min((aim[1] for s in sorted([p for p in world.my_planets], key=lambda p: dist(p.x, p.y, t.x, t.y))[:REACTION_SOURCE_TOP_K_MY] if (sd:=world.best_probe_aim(s.id, t.id, max(1, int(s.ships)))) and (aim:=sd[1])), default=10**9),
                min((aim[1] for s in sorted([p for p in world.enemy_planets], key=lambda p: dist(p.x, p.y, t.x, t.y))[:REACTION_SOURCE_TOP_K_ENEMY] if (sd:=world.best_probe_aim(s.id, t.id, max(1, int(s.ships)))) and (aim:=sd[1])), default=10**9)
            )

    for p in world.my_planets:
        if exp(): break
        pkeep = max((int(e.ships * PROACTIVE_DEFENSE_RATIO) for e in sorted(world.enemy_planets, key=lambda en: dist(en.x, en.y, p.x, p.y))[:PROACTIVE_ENEMY_TOP_K] if (sd:=world.plan_shot(e.id, p.id, max(1, int(e.ships)))) and sd[1] <= PROACTIVE_DEFENSE_HORIZON), default=0)
        pkeep = max(pkeep, stacked_enemy_proactive_keep(p, world))
        
        if p.id == hub_id:
            hub_floating_reserve = int(p.ships * (HUB_RESERVE_INNER if world.is_inner_hub else HUB_RESERVE_OUTER))
            pkeep = max(pkeep, hub_floating_reserve)

        res[p.id] = min(int(p.ships), max(world.keep_needed_map.get(p.id, 0), pkeep))
        atk[p.id] = max(0, int(p.ships) - res[p.id])

    return {"indirect_wealth_map": iwm, "reserve": res, "attack_budget": atk, "reaction_time_map": rtm, "hub_id": hub_id}

def build_modes(world):
    dom = (world.my_total - world.enemy_total) / max(1, world.my_total + world.enemy_total)
    ib, ia = dom < BEHIND_DOMINATION, dom > AHEAD_DOMINATION
    amm = 1.0 + (AHEAD_ATTACK_MARGIN_BONUS if ia else 0) - (BEHIND_ATTACK_MARGIN_PENALTY if ib else 0)
    isf = dom > FINISHING_DOMINATION and world.my_prod > world.enemy_prod * FINISHING_PROD_RATIO and world.step > 80
    if isf: amm += FINISHING_ATTACK_MARGIN_BONUS
    return {"domination": dom, "is_behind": ib, "is_ahead": ia, "is_dominating": ia or (world.max_enemy_strength > 0 and world.my_total > world.max_enemy_strength * 1.25), "is_finishing": isf, "attack_margin_mult": amm}

def target_value(target, arrival_turns, mission, world, modes, policy):
    tp = max(1, world.remaining_steps - arrival_turns)
    if target.id in world.comet_ids:
        tp = max(0, min(tp, world.comet_life(target.id) - arrival_turns))
        if tp <= 0: return -1.0

    val = target.production * tp + policy["indirect_wealth_map"][target.id] * tp * INDIRECT_VALUE_SCALE
    target_ring = orbital_radius(target)

    ring_diff = abs(target_ring - world.avg_radius)
    if ring_diff < ORBITAL_ARC_TOLERANCE:
        val *= ORBITAL_ARC_BONUS 
        
    if world.is_outer_hub and target_ring < world.avg_radius - 5.0:
        val *= LEAPFROG_FOB_BONUS 

    if world.is_static(target.id): val *= STATIC_NEUTRAL_VALUE_MULT if target.owner == -1 else STATIC_HOSTILE_VALUE_MULT
    else: val *= ROTATING_OPENING_VALUE_MULT if world.is_opening else 1.0

    if target.owner not in (-1, world.player):
        val *= OPENING_HOSTILE_TARGET_VALUE_MULT if world.is_opening else HOSTILE_TARGET_VALUE_MULT
        if world.owner_strength.get(target.owner, 0) <= WEAK_ENEMY_THRESHOLD: val += ELIMINATION_BONUS
        if world._weakest_enemy == target.owner: val *= WEAKEST_ENEMY_VALUE_MULT_4P if world.is_four_player else WEAKEST_ENEMY_VALUE_MULT_2P
    elif target.owner == -1:
        my_t, en_t = policy["reaction_time_map"].get(target.id, (10**9, 10**9))
        if my_t <= en_t - SAFE_NEUTRAL_MARGIN: val *= SAFE_NEUTRAL_VALUE_MULT * (1.08 if modes["is_behind"] else 1.0)
        elif abs(my_t - en_t) <= CONTESTED_NEUTRAL_MARGIN: val *= CONTESTED_NEUTRAL_VALUE_MULT * (0.92 if modes["is_dominating"] else 1.0)
        if world.is_early: val *= EARLY_NEUTRAL_VALUE_MULT
        if modes["is_behind"] and not world.is_static(target.id): val *= BEHIND_ROTATING_NEUTRAL_VALUE_MULT

    if target.id in world.comet_ids: val *= COMET_VALUE_MULT
    if mission == "snipe": val *= SNIPE_VALUE_MULT
    elif mission == "swarm": val *= SWARM_VALUE_MULT
    elif mission == "crash_exploit": val *= CRASH_EXPLOIT_VALUE_MULT
    elif mission == "gang_up": val *= GANG_UP_VALUE_MULT

    if target.id in world.exposed_planet_ids: val *= EXPOSED_PLANET_VALUE_MULT
    if world.is_late: val += max(0, target.ships) * LATE_IMMEDIATE_SHIP_VALUE
    if modes["is_finishing"] and target.owner not in (-1, world.player): val *= FINISHING_HOSTILE_VALUE_MULT

    return val

def preferred_send(target, base_needed, arrival_turns, src_available, world, modes, policy):
    send = max(base_needed, int(math.ceil(base_needed * modes["attack_margin_mult"])))
    margin = (NEUTRAL_MARGIN_BASE + target.production * NEUTRAL_MARGIN_PROD_WEIGHT if target.owner == -1 else HOSTILE_MARGIN_BASE + target.production * HOSTILE_MARGIN_PROD_WEIGHT)
    margin = min(NEUTRAL_MARGIN_CAP if target.owner == -1 else HOSTILE_MARGIN_CAP, margin)
    
    if world.is_static(target.id): margin += STATIC_TARGET_MARGIN
    my_t, en_t = policy["reaction_time_map"].get(target.id, (10**9, 10**9))
    if target.owner == -1 and abs(my_t - en_t) <= CONTESTED_NEUTRAL_MARGIN: margin += CONTESTED_TARGET_MARGIN
    if world.is_four_player: margin += FOUR_PLAYER_TARGET_MARGIN
    if arrival_turns > LONG_TRAVEL_MARGIN_START: margin += min(LONG_TRAVEL_MARGIN_CAP, arrival_turns // LONG_TRAVEL_MARGIN_DIVISOR)
    if target.id in world.comet_ids: margin = max(0, margin - COMET_MARGIN_RELIEF)
    if modes["is_finishing"] and target.owner not in (-1, world.player): margin += FINISHING_HOSTILE_SEND_BONUS
    if target.id in world.exposed_planet_ids or (world.is_four_player and world._weakest_enemy == target.owner): margin = max(0, margin - 2)
    return min(src_available, send + margin)

def settle_plan(src, target, src_cap, send_guess, world, pc, modes, policy, mission="capture", eval_turn_fn=lambda t: t, anchor_turn=None, anchor_tolerance=None, max_iter=4):
    if src_cap < 1: return None
    seed = max(1, min(src_cap, int(send_guess)))
    anchor_tolerance = anchor_tolerance if anchor_tolerance is not None else (1 if mission == "snipe" else None)
    tested, order = {},[]

    def ev(s):
        if s in tested: return tested[s]
        aim = world.plan_shot(src.id, target.id, s)
        if not aim or (mission == "crash_exploit" and anchor_turn and aim[1] < anchor_turn) or int(math.ceil(eval_turn_fn(aim[1]))) < aim[1]:
            tested[s] = None; return None
        need = world.min_ships_to_own_by(target.id, int(math.ceil(eval_turn_fn(aim[1]))), world.player, arrival_turn=aim[1], planned_commitments=pc, upper_bound=src_cap)
        if need <= 0 or need > src_cap:
            tested[s] = None; return None
        des = need if mission in ("snipe", "crash_exploit") else min(src_cap, max(need, need + DEFENSE_SEND_MARGIN_BASE + target.production * DEFENSE_SEND_MARGIN_PROD_WEIGHT if mission == "rescue" else preferred_send(target, need, aim[1], src_cap, world, modes, policy)))
        tested[s] = (aim[0], aim[1], int(math.ceil(eval_turn_fn(aim[1]))), need, s, des)
        order.append(s); return tested[s]

    cs = next((s for s in world.probe_ship_candidates(src.id, target.id, src_cap, (seed,)) if (r:=ev(s)) and not (anchor_turn and anchor_tolerance and abs(r[1] - anchor_turn) > anchor_tolerance)), None)
    if not cs: return None

    for _ in range(max_iter):
        r = ev(cs)
        if not r: break
        if r[5] == r[4]:
            if (anchor_turn and anchor_tolerance and abs(r[1] - anchor_turn) > anchor_tolerance) or (mission == "rescue" and r[1] > r[2]): return None
            return r[:5]
        cs = max(1, min(src_cap, int(r[5])))
        if cs in tested: break

    for s in sorted([s for s in order if tested[s]], key=lambda s: (abs(tested[s][1] - anchor_turn) if mission == "snipe" and anchor_turn else 0, abs(s - seed), tested[s][1], s)):
        r = tested[s]
        if r[4] >= r[3] and not (anchor_turn and anchor_tolerance and abs(r[1] - anchor_turn) > anchor_tolerance) and not (mission == "rescue" and r[1] > r[2]): return r[:5]
    return None

# ============================================================
# Core Planners
# ============================================================

def build_gang_up_missions(world, policy, pc, modes):
    missions =[]
    for b in detect_enemy_planet_battles(world.arrivals_by_planet, world.enemy_planets, world.player):
        tgt = world.planet_by_id[b["target_id"]]
        da = b["battle_turn"] + GANG_UP_POST_BATTLE_DELAY
        for src in world.my_planets:
            if (sa := policy["attack_budget"].get(src.id, 0)) < PARTIAL_SOURCE_MIN_SHIPS: continue
            if not (sd := world.best_probe_aim(src.id, tgt.id, sa, (max(3, b["post_ships"] + 3), int(tgt.ships) + 1), anchor_turn=da, max_anchor_diff=GANG_UP_ETA_WINDOW)): continue
            if not (plan := settle_plan(src, tgt, sa, sd[0], world, pc, modes, policy, "gang_up", lambda t: max(t, da), da, GANG_UP_ETA_WINDOW)): continue
            if plan[1] > world.remaining_steps - LATE_CAPTURE_BUFFER: continue
            if (v := target_value(tgt, plan[1], "gang_up", world, modes, policy)) > 0:
                cost_weight = ATTACK_COST_TURN_WEIGHT * (1.2 if world.is_outer_hub else 1.0)
                sc = v / (plan[4] + plan[1] * cost_weight + 1.0)
                missions.append(Mission("single", sc, tgt.id, plan[1],[ShotOption(sc, src.id, tgt.id, plan[0], plan[1], plan[3], plan[4], "gang_up", da)]))
    return missions

def build_elimination_missions(world, policy, pc, modes):
    if not world._weakest_enemy or world._weakest_enemy_strength > world.my_total * 0.9: return []
    others =[s for o, s in world.owner_strength.items() if o not in (world.player, world._weakest_enemy)]
    if others and world._weakest_enemy_strength > min(others) * 0.95: return []
    
    missions =[]
    mult = 1.5 if world.is_four_player else 1.25
    for tgt in[p for p in world.enemy_planets if p.owner == world._weakest_enemy]:
        for src in world.my_planets:
            if (sa := policy["attack_budget"].get(src.id, 0)) < PARTIAL_SOURCE_MIN_SHIPS: continue
            if not (sd := world.best_probe_aim(src.id, tgt.id, sa, (int(tgt.ships) + 1, int(tgt.ships) + 5))): continue
            if sd[1][1] > world.remaining_steps - LATE_CAPTURE_BUFFER: continue
            need = world.min_ships_to_own_at(tgt.id, sd[1][1], world.player, pc)
            if need <= 0 or need > sa: continue
            if not (plan := settle_plan(src, tgt, sa, preferred_send(tgt, need, sd[1][1], sa, world, modes, policy), world, pc, modes, policy)): continue
            if plan[1] > world.remaining_steps - LATE_CAPTURE_BUFFER or plan[4] < plan[3]: continue
            if (v := target_value(tgt, plan[1], "capture", world, modes, policy)) > 0:
                cost_weight = ATTACK_COST_TURN_WEIGHT * (1.2 if world.is_outer_hub else 1.0)
                sc = (v * mult) / (plan[4] + plan[1] * cost_weight + 1.0)
                missions.append(Mission("single", sc, tgt.id, plan[1], [ShotOption(sc, src.id, tgt.id, plan[0], plan[1], plan[3], plan[4])]))
    return missions

def plan_moves(world, deadline=None):
    exp = lambda: deadline and time.perf_counter() > deadline
    hp = lambda: (deadline is None or deadline - time.perf_counter() > HEAVY_PHASE_MIN_TIME) and len(world.planets) <= HEAVY_ROUTE_PLANET_LIMIT
    
    modes, policy = build_modes(world), build_policy_state(world, deadline)
    pc, sopt, mis, mov, spent = defaultdict(list), defaultdict(list), [],[], defaultdict(int)
    
    sleft = lambda sid: max(0, int(world.planet_by_id[sid].ships) - spent[sid])
    aleft = lambda sid: max(0, policy["attack_budget"].get(sid, 0) - spent[sid])
    
    def addm(sid, a, s):
        sd = min(int(s), sleft(sid))
        if sd >= 1: mov.append([sid, float(a), sd]); spent[sid] += sd
        return sd

    # 1. High Priority Missions (Gang-ups & Crashes)
    if hp():
        mis.extend(build_gang_up_missions(world, policy, pc, modes))
        mis.extend(build_elimination_missions(world, policy, pc, modes))
        for cr in detect_enemy_crashes(world):
            tgt = world.planet_by_id[cr["target_id"]]
            da = cr["crash_turn"] + CRASH_EXPLOIT_POST_CRASH_DELAY
            for src in world.my_planets:
                if (sa := aleft(src.id)) < PARTIAL_SOURCE_MIN_SHIPS: continue
                if not (sd := world.best_probe_aim(src.id, tgt.id, sa, (12, int(tgt.ships) + 1), anchor_turn=da, max_anchor_diff=CRASH_EXPLOIT_ETA_WINDOW)): continue
                if not (plan := settle_plan(src, tgt, sa, sd[0], world, pc, modes, policy, "crash_exploit", lambda t: max(t, da), da, CRASH_EXPLOIT_ETA_WINDOW)): continue
                if plan[1] > world.remaining_steps - LATE_CAPTURE_BUFFER: continue
                if (v := target_value(tgt, plan[1], "crash_exploit", world, modes, policy)) > 0:
                    cost_weight = SNIPE_COST_TURN_WEIGHT * (1.2 if world.is_outer_hub else 1.0)
                    sc = v / (plan[4] + plan[1] * cost_weight + 1.0)
                    mis.append(Mission("crash_exploit", sc, tgt.id, plan[1],[ShotOption(sc, src.id, tgt.id, plan[0], plan[1], plan[3], plan[4], "crash_exploit", da)]))

    # 2. Standard Captures & Swarms
    for src in world.my_planets:
        if exp(): break
        if (sa := aleft(src.id)) <= 0: continue
        
        cost_weight = ATTACK_COST_TURN_WEIGHT * (1.25 if world.is_outer_hub else 1.0)

        for tgt in world.planets:
            if tgt.id == src.id or tgt.owner == world.player: continue
            if not (sd := world.best_probe_aim(src.id, tgt.id, sa, (int(tgt.ships) + 1,))): continue
            rt = sd[1][1]
            if rt > world.remaining_steps - (VERY_LATE_CAPTURE_BUFFER if world.is_very_late else LATE_CAPTURE_BUFFER): continue
            gn = world.min_ships_to_own_at(tgt.id, rt, world.player, pc)
            if gn <= 0 or (world.is_opening and tgt.owner == -1 and tgt.production < SAFE_OPENING_PROD_THRESHOLD and rt > ROTATING_OPENING_MAX_TURNS): continue
            
            psc = min(sa, preferred_send(tgt, gn, rt, sa, world, modes, policy))
            if psc >= PARTIAL_SOURCE_MIN_SHIPS and (ps := world.best_probe_aim(src.id, tgt.id, psc, (psc, gn, int(tgt.ships)+1))) and (v := target_value(tgt, ps[1][1], "swarm", world, modes, policy)) > 0:
                sc = v / (psc + ps[1][1] * cost_weight + 1.0)
                sopt[tgt.id].append(ShotOption(sc, src.id, tgt.id, ps[1][0], ps[1][1], gn, psc, "swarm"))

            if gn <= sa and (plan := settle_plan(src, tgt, sa, preferred_send(tgt, gn, rt, sa, world, modes, policy), world, pc, modes, policy)) and plan[4] >= 1 and (v := target_value(tgt, plan[1], "capture", world, modes, policy)) > 0:
                sc = v / (plan[4] + plan[1] * cost_weight + 1.0)
                mis.append(Mission("single", sc, tgt.id, plan[1], [ShotOption(sc, src.id, tgt.id, plan[0], plan[1], plan[3], plan[4])]))

    for tgt_id, opts in sopt.items():
        if exp(): break
        if len(opts) < 2: continue
        tgt = world.planet_by_id[tgt_id]
        top = sorted(opts, key=lambda x: -x.score)[:MULTI_SOURCE_TOP_K]
        tol = HOSTILE_SWARM_ETA_TOLERANCE if tgt.owner not in (-1, world.player) else MULTI_SOURCE_ETA_TOLERANCE
        
        for i in range(len(top)):
            for j in range(i+1, len(top)):
                if top[i].src_id == top[j].src_id or abs(top[i].turns - top[j].turns) > (tol if len(top)<3 else THREE_SOURCE_ETA_TOLERANCE): continue
                jt = max(top[i].turns, top[j].turns)
                nd = world.min_ships_to_own_at(tgt_id, jt, world.player, pc, upper_bound=top[i].send_cap + top[j].send_cap)
                if nd > 0 and top[i].send_cap < nd and top[j].send_cap < nd and top[i].send_cap + top[j].send_cap >= nd and (v := target_value(tgt, jt, "swarm", world, modes, policy)) > 0:
                    mis.append(Mission("swarm", (v / (nd + jt * ATTACK_COST_TURN_WEIGHT + 1.0)) * MULTI_SOURCE_PLAN_PENALTY, tgt_id, jt, [top[i], top[j]]))

    mis.sort(key=lambda x: -x.score)
    for m in mis:
        if exp(): break
        tgt = world.planet_by_id[m.target_id]
        if m.kind in ("single", "snipe", "crash_exploit", "gang_up"):
            o = m.options[0]
            if (left := aleft(o.src_id)) <= 0: continue
            plan = settle_plan(world.planet_by_id[o.src_id], tgt, left, min(left, o.send_cap), world, pc, modes, policy, m.kind, lambda t, a=o.anchor_turn: max(t, a) if a else t, o.anchor_turn, CRASH_EXPLOIT_ETA_WINDOW if m.kind=="crash_exploit" else (GANG_UP_ETA_WINDOW if m.kind=="gang_up" else (1 if m.kind=="snipe" else None)))
            if plan and plan[4] >= plan[3] and plan[3] <= left and (sent := addm(o.src_id, plan[0], plan[4])) >= plan[3]:
                pc[tgt.id].append((plan[1], world.player, int(sent)))
        else: # Swarms
            lims =[min(aleft(o.src_id), o.send_cap) for o in m.options]
            if min(lims) <= 0 or (miss := world.min_ships_to_own_at(tgt.id, m.turns, world.player, pc, upper_bound=sum(lims))) <= 0 or sum(lims) < miss: continue
            ord_opts = sorted(zip(m.options, lims), key=lambda x: (x[0].turns, -x[1], x[0].src_id))
            rem, snd = miss, {}
            for idx, (o, l) in enumerate(ord_opts):
                s = min(l, max(0, rem - sum(ol for _, ol in ord_opts[idx+1:])))
                snd[o.src_id], rem = s, rem - s
            if rem > 0: continue
            ream =[(o.src_id, (aim:=world.plan_shot(o.src_id, tgt.id, snd[o.src_id]))[0], aim[1], snd[o.src_id]) for o, _ in ord_opts if snd.get(o.src_id, 0) > 0 and world.plan_shot(o.src_id, tgt.id, snd[o.src_id])]
            if not ream or max(t for _,_,t,_ in ream) - min(t for _,_,t,_ in ream) > (HOSTILE_SWARM_ETA_TOLERANCE if tgt.owner not in (-1, world.player) else MULTI_SOURCE_ETA_TOLERANCE) or world.projected_state(tgt.id, max(t for _,_,t,_ in ream), pc,[(t, world.player, s) for _,_,t,s in ream])[0] != world.player: continue
            com =[(t, world.player, int(act)) for sid, a, t, s in ream if (act := addm(sid, a, s)) > 0]
            if sum(s for _,_,s in com) >= miss: pc[tgt.id].extend(com)

    # 4. Supply Chain Logistics
    if not world.is_late and (world.enemy_planets or world.neutral_planets) and len(world.my_planets) > 1 and (deadline is None or deadline - time.perf_counter() > 0.08):
        f_tgts = world.enemy_planets or (world.static_neutral_planets or world.neutral_planets)
        f_dist = {p.id: nearest_distance_to_set(p.x, p.y, f_tgts) for p in world.my_planets}
        
        hub_id = policy.get("hub_id")
        hub_planet = world.planet_by_id[hub_id] if hub_id else None

        for r in sorted(world.my_planets, key=lambda p: -f_dist[p.id]):
            if exp(): break
            if aleft(r.id) < REAR_SOURCE_MIN_SHIPS: continue
            
            front = None
            if world.is_outer_hub and hub_planet and r.id != hub_id and f_dist[r.id] > f_dist[hub_id]:
                front = hub_planet
            else:
                s_fronts =[p for p in world.my_planets if not (world.hold_status(p.id, pc, DOOMED_EVAC_TURN_LIMIT).get("fall_turn") and sleft(p.id) >= DOOMED_MIN_SHIPS)]
                if s_fronts:
                    anch = min(s_fronts, key=lambda p: f_dist[p.id])
                    if r.id != anch.id and f_dist[r.id] >= f_dist[anch.id] * REAR_DISTANCE_RATIO:
                        scands =[p for p in s_fronts if p.id != r.id and f_dist[p.id] < f_dist[r.id] * REAR_STAGE_PROGRESS]
                        if scands:
                            front = min(scands, key=lambda p: dist(r.x, r.y, p.x, p.y))

            if front and front.id != r.id:
                srat = REAR_SEND_RATIO_FOUR_PLAYER if world.is_four_player else REAR_SEND_RATIO_TWO_PLAYER
                snd = int(aleft(r.id) * srat)
                if snd >= REAR_SEND_MIN_SHIPS and (aim := world.plan_shot(r.id, front.id, snd)) and aim[1] <= REAR_MAX_TRAVEL_TURNS:
                    addm(r.id, aim[0], snd)

    fm, uf =[], defaultdict(int)
    for sid, a, s in mov:
        if (ms := min(s, int(world.planet_by_id[sid].ships) - uf[sid])) >= 1:
            fm.append([sid, float(a), int(ms)]); uf[sid] += ms
    return fm

# ============================================================
# Agent Entry Point
# ============================================================
_agent_step = 0

def agent(obs, config=None):
    global _agent_step
    _agent_step += 1
    
    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    obs_step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    step = max(obs_step, _agent_step - 1)
    raw_planets = obs.get("planets", []) if is_dict else getattr(obs, "planets",[])
    raw_fleets = obs.get("fleets",[]) if is_dict else getattr(obs, "fleets",[])
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    comets = obs.get("comets", []) if is_dict else getattr(obs, "comets",[])
    comet_ids = obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids",[])
    
    act_timeout = 1.0
    if config is not None:
        act_timeout = config.get("actTimeout", 1.0) if isinstance(config, dict) else getattr(config, "actTimeout", 1.0)
        
    w = WorldModel(
        player, step, [Planet(*p) for p in raw_planets],
        [Fleet(*f) for f in raw_fleets],
        ang_vel, comets, comet_ids
    )
    
    if not w.my_planets: return[]
        
    deadline = time.perf_counter() + min(SOFT_ACT_DEADLINE, max(0.55, act_timeout * 0.82))
    return plan_moves(w, deadline=deadline)

__all__ = ["agent"]

Writing main.py
